# Edit Distance, Spelling Correction and OOV Handling

Day 1, Session 4. Measuring how different two strings are, fixing typos, and coping with words we have never seen

Run this in Google Colab (or any Jupyter setup). Run the cells top to bottom.

## Setup

In [ ]:
# one time installs, colab already has most of these
# nltk   -> brown corpus (text for word counts), nltk.edit_distance for checking our own code
# pandas -> to print the edit distance table nicely
%pip install -q nltk pandas

In [ ]:
import math
import nltk
import pandas as pd
from collections import Counter

# brown is a general english corpus, about a million words
nltk.download("brown", quiet=True)
from nltk.corpus import brown

# word counts, lowercase, letters only
counts = Counter(w.lower() for w in brown.words() if w.isalpha())
total = sum(counts.values())
print(len(counts), "distinct words,", total, "tokens")

## 1. Spotting non-word errors

The simplest detector: is the word in our vocabulary?

In [ ]:
vocab = set(counts)
for w in ["teh", "there", "speling", "nlp"]:
    print(w, w in vocab)

# 'nlp' is flagged even though it is a fine word, the corpus is from 1961
# 'there' passes, but a dictionary cannot tell that "there going home" is wrong

## 2. Minimum edit distance

Fill a table where cell (i, j) is the distance between a[:i] and b[:j].

In [ ]:
def edit_distance(a, b):
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        d[i][0] = i                      # turning a[:i] into empty string = i deletions
    for j in range(len(b) + 1):
        d[0][j] = j                      # empty string into b[:j] = j insertions
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,          # delete
                          d[i][j - 1] + 1,          # insert
                          d[i - 1][j - 1] + cost)   # substitute or match
    return d[-1][-1]

print(edit_distance("kitten", "sitting"))
print(edit_distance("sunday", "saturday"))

In [ ]:
# print the whole table to trace the path
def edit_table(a, b):
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        d[i][0] = i
    for j in range(len(b) + 1):
        d[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + cost)
    return pd.DataFrame(d, index=["#"] + list(a), columns=["#"] + list(b))

print(edit_table("kitten", "sitting"))

In [ ]:
# check against nltk, and see the transposition difference
from nltk.metrics import edit_distance as nltk_ed

print(nltk_ed("kitten", "sitting"))
print("teh -> the, levenshtein:", nltk_ed("teh", "the"))
print("teh -> the, with transposition:", nltk_ed("teh", "the", transpositions=True))

## 3. Spelling correction

Generate everything one edit away, keep real words, pick the most frequent.

In [ ]:
letters = "abcdefghijklmnopqrstuvwxyz"

def edits1(w):
    splits = [(w[:i], w[i:]) for i in range(len(w) + 1)]
    deletes = [a + b[1:] for a, b in splits if b]
    transposes = [a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1]
    replaces = [a + c + b[1:] for a, b in splits if b for c in letters]
    inserts = [a + c + b for a, b in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

def edits2(w):
    return {e2 for e1 in edits1(w) for e2 in edits1(e1)}

def correct(w):
    if w in counts:
        return w
    cands = [x for x in edits1(w) if x in counts] or [x for x in edits2(w) if x in counts] or [w]
    return max(cands, key=counts.get)

print(len(edits1("speling")), "strings one edit away from 'speling'")
cands = [w for w in edits1("speling") if w in counts]
print(sorted(cands, key=counts.get, reverse=True))
print(correct("teh"), correct("recieve"), correct("helo"))

In [ ]:
# evaluate on a small hand made test set
pairs = [("teh", "the"), ("recieve", "receive"), ("definately", "definitely"),
         ("seperate", "separate"), ("occured", "occurred"), ("wich", "which"),
         ("thier", "their"), ("freind", "friend"), ("beleive", "believe"),
         ("adress", "address"), ("tommorow", "tomorrow"), ("accomodate", "accommodate"),
         ("untill", "until"), ("goverment", "government"), ("enviroment", "environment"),
         ("helo", "hello"), ("speling", "spelling"), ("colege", "college"),
         ("mesage", "message"), ("studnet", "student")]

right = 0
for typo, truth in pairs:
    guess = correct(typo)
    right += guess == truth
    if guess != truth:
        print("missed:", typo, "->", guess, "(wanted", truth + ")")
print(right, "of", len(pairs), "correct")

## 4. Noisy channel scoring

score = log P(w) + edits * log(penalty). Watch what the penalty does.

In [ ]:
def show(typo, cands, penalty):
    print("penalty", penalty)
    for w in cands:
        d = edit_distance(typo, w)
        prior = math.log(counts[w] / total)
        print("  ", w, "edits", d, "score", round(prior + d * math.log(penalty), 1))

show("teh", ["the", "ten", "tea"], 0.01)
show("teh", ["the", "ten", "tea"], 0.001)

# with 0.001 'ten' wins, which is wrong. an error model that counts a swap as one edit would fix it

## 5. Out-of-vocabulary words

In [ ]:
train = [w.lower() for w in brown.words(categories="news") if w.isalpha()]
test = [w.lower() for w in brown.words(categories="science_fiction") if w.isalpha()]
seen = set(train)
oov = [w for w in test if w not in seen]
print(len(test), len(oov), round(100 * len(oov) / len(test), 1))
print(Counter(oov).most_common(10))

In [ ]:
# unk replacement: rare training words also become <UNK>, so the model has seen it
freq = Counter(train)
vocab_unk = {w for w, c in freq.items() if c >= 2}

def to_unk(tokens):
    return [t if t in vocab_unk else "<UNK>" for t in tokens]

print(to_unk("the spaceship zoomed past nlp students".split()))

In [ ]:
# preview of subwords: an unknown word is cut into smaller pieces
# this tiny version uses character trigrams, bert uses wordpiece and gpt uses bpe (day 3)
def char_ngrams(word, n=3):
    padded = "<" + word + ">"
    return [padded[i:i + n] for i in range(len(padded) - n + 1)]

print(char_ngrams("zoomed"))
print(char_ngrams("unfriendly"))

## 6. Putting it together

In [ ]:
def fix(sentence):
    out = []
    for w in sentence.split():
        out.append(correct(w.lower()) if w.isalpha() else w)
    return " ".join(out)

print(fix("I recieved a mesage from teh colege"))

# a real-word error slips through, nothing here looks at context
print(fix("there going home"))

## Practice

1. (Easy) Compute by hand, with the table method, the edit distance between 'cat' and 'cart'. Check it with edit_distance.

2. (Medium) Extend correct() so that it also tries splitting a word into two real words, for example 'thecat' to 'the cat'.

3. (Challenge) Evaluate correct() on at least 20 typo and correct word pairs. Report accuracy and list the failures. Then add the language prior from the previous word to fix one failure.

In [ ]:
# your practice code here